In [4]:
# Ячейка 1
import torch
print("torch:", torch.__version__)
print("CUDA доступна:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "нет GPU")
!nvidia-smi --query-gpu=name,memory.total --format=csv

torch: 2.11.0+cu128
CUDA доступна: True
GPU: Tesla T4
name, memory.total [MiB]
Tesla T4, 15360 MiB


In [5]:
# Ячейка 2
!pip install -q datasets

In [6]:
# Ячейка 3
# Монтируем Drive СРАЗУ (чекпойнты понадобятся на шаге 6, но папку лучше завести с первого дня, чтобы потом не переделывать пути)
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/sasrec'
os.makedirs(PROJECT_DIR, exist_ok=True)
print("Файлы проекта будут тут:", PROJECT_DIR)

Mounted at /content/drive
Файлы проекта будут тут: /content/drive/MyDrive/sasrec


## 1. Данные: загрузка Yambda likes

In [30]:
# Ячейка 4 загрузка лайков
import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"   # без этого Colab оставляет виджет-метаданные,
                                                   # и GitHub потом не может отрендерить ноутбук
from datasets import load_dataset
from datasets.utils.logging import disable_progress_bar
disable_progress_bar()

ds = load_dataset("yandex/yambda", data_dir="flat/50m", data_files="likes.parquet")
likes = ds["train"]               # load_dataset всегда отдаёт DatasetDict, берём единственный сплит
print(likes)                      # покажет колонки и число строк

Dataset({
    features: ['uid', 'timestamp', 'item_id', 'is_organic'],
    num_rows: 881456
})


In [8]:
# Ячейка 5
import pandas as pd

df = likes.to_pandas()
print("Форма (строк, колонок):", df.shape)
display(df.head())

n_users = df['uid'].nunique()
n_items = df['item_id'].nunique()
print(f"\nПользователей: {n_users:,}")
print(f"Уникальных треков в likes: {n_items:,}")
print(f"Всего событий: {len(df):,}")

# Распределение длины истории
# Оно подскажет, какую максимальную длину последовательности брать
hist_len = df.groupby('uid').size()
print("\nДлина истории на пользователя:")
print(hist_len.describe(percentiles=[.25, .5, .75, .9, .95, .99]))

# Санити-чек: данные действительно отсортированы по (uid, timestamp)?
sorted_check = df.equals(df.sort_values(['uid', 'timestamp']).reset_index(drop=True))
print("\nОтсортировано по (uid, timestamp):", sorted_check)

Форма (строк, колонок): (881456, 4)


,uid,timestamp,item_id,is_organic
0,100,44755,732449,1
1,100,1155860,6568592,0
2,100,1259125,5411243,1
3,100,1260005,7371186,0
4,100,1263935,4943655,0



Пользователей: 8,283
Уникальных треков в likes: 181,304
Всего событий: 881,456

Длина истории на пользователя:
count    8283.000000
mean      106.417482
std       209.325608
min         1.000000
25%        10.000000
50%        44.000000
75%       124.000000
90%       269.800000
95%       398.000000
99%       847.080000
max      8697.000000
dtype: float64

Отсортировано по (uid, timestamp): True


## 2. Подготовка: маппинг, фильтр, leave-last-out split

In [9]:
# Ячейка 6
# Вводим параметры

MIN_LEN = 5      # минимум событий у юзера: 2 уходят на val+test, на train остаётся >= 3
MAX_LEN = 200    # максимальная длина последовательности (по перцентилям истории)
PAD_IDX = 0      # индекс-заглушка под паддинг коротких историй

In [10]:
# Ячейка 7
# выкидываем слишком коротких юзеров
before = df['uid'].nunique()
sizes = df.groupby('uid')['item_id'].transform('size')   # размер группы для каждой строки
df = df[sizes >= MIN_LEN].copy()
after = df['uid'].nunique()
print(f"Юзеров было: {before:,}, осталось (>= {MIN_LEN} событий): {after:,}")

Юзеров было: 8,283, осталось (>= 5 событий): 6,951


In [11]:
# Ячейка 8
unique_items = df['item_id'].unique()
item2idx = {int(it): i for i, it in enumerate(unique_items, start=1)}  # старт с 1!
idx2item = {i: it for it, i in item2idx.items()}

n_items = len(item2idx)
vocab_size = n_items + 1          # +1 на PAD; это и есть число строк embedding-таблицы
print(f"Уникальных треков: {n_items:,}")
print(f"Размер словаря (вместе с PAD): {vocab_size:,}")

df['idx'] = df['item_id'].map(item2idx).astype('int64')   # переводим колонку в индексы

Уникальных треков: 180,942
Размер словаря (вместе с PAD): 180,943


In [12]:
# Ячейка 9
# Последовательности + leave-last-out
df = df.sort_values(['uid', 'timestamp'])        # подстраховка, хоть уже и отсортировано
user_seqs = df.groupby('uid')['idx'].apply(list)

train_seqs = {}
val_data   = {}
test_data  = {}

for uid, seq in user_seqs.items():
    train_seq = seq[:-2][-MAX_LEN:]              # вся история без последних двух, обрезано до MAX_LEN
    train_seqs[uid] = train_seq
    val_data[uid]  = (train_seq,           seq[-2])   # тот же вход -> предсказать предпоследний
    test_data[uid] = (seq[:-1][-MAX_LEN:], seq[-1])   # история без последнего -> предсказать последний

# наглядный sanity-check на одном юзере
u = next(iter(train_seqs))
print("uid:", u)
print("train_seq (последние 8 idx):", train_seqs[u][-8:])
print("val : вход кончается на", val_data[u][0][-1], "-> предсказать", val_data[u][1])
print("test: вход кончается на", test_data[u][0][-1], "-> предсказать", test_data[u][1])
print("\nЮзеров в сплите:", len(train_seqs))

uid: 100
train_seq (последние 8 idx): [11, 12, 13, 14, 15, 16, 17, 18]
val : вход кончается на 18 -> предсказать 19
test: вход кончается на 19 -> предсказать 20

Юзеров в сплите: 6951


In [13]:
# Ячейка 10
import pickle

artifacts = {
    'item2idx': item2idx, 'idx2item': idx2item, 'vocab_size': vocab_size,
    'MAX_LEN': MAX_LEN, 'PAD_IDX': PAD_IDX,
    'train_seqs': train_seqs, 'val_data': val_data, 'test_data': test_data,
}
save_path = f"{PROJECT_DIR}/data_likes.pkl"
with open(save_path, 'wb') as f:
    pickle.dump(artifacts, f)
print("Сохранено:", save_path)

Сохранено: /content/drive/MyDrive/sasrec/data_likes.pkl


In [14]:
# Ячейка 10.2 восстановление данных из Drive (после реконнекта Colab)
# Если рантайм отвалился выполняем только эту ячейку, и можно сразу прыгать к шагу 3,
# не пересобирая словарь и сплит с нуля.
import pickle
with open(f"{PROJECT_DIR}/data_likes.pkl", "rb") as f:
    a = pickle.load(f)
item2idx, idx2item = a['item2idx'], a['idx2item']
vocab_size, MAX_LEN, PAD_IDX = a['vocab_size'], a['MAX_LEN'], a['PAD_IDX']
train_seqs, val_data, test_data = a['train_seqs'], a['val_data'], a['test_data']
n_items = len(item2idx)
print(f"Восстановлено: {len(train_seqs):,} юзеров, {n_items:,} треков")

Восстановлено: 6,951 юзеров, 180,942 треков


## 3. Dataset и DataLoader с паддингом

In [15]:
# Ячейка 11
import torch
from torch.utils.data import Dataset, DataLoader

class SeqDataset(Dataset):
    def __init__(self, user_seqs_dict, max_len, pad_idx=0):
        self.seqs = list(user_seqs_dict.values())   # uid в модели не нужен, берём только последовательности
        self.max_len = max_len
        self.pad_idx = pad_idx

    def __len__(self):
        return len(self.seqs)

    def __getitem__(self, i):
        s = self.seqs[i]
        inp = s[:-1][-self.max_len:]
        tgt = s[1:][-self.max_len:]

        pad = self.max_len - len(inp)
        inp = [self.pad_idx] * pad + inp
        tgt = [self.pad_idx] * pad + tgt

        return (torch.tensor(inp, dtype=torch.long),
                torch.tensor(tgt, dtype=torch.long))

In [16]:
# Ячейка 12
BATCH_SIZE = 128

train_ds = SeqDataset(train_seqs, MAX_LEN, PAD_IDX)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, drop_last=True)
print("Примеров в train:", len(train_ds))
print("Батчей за эпоху:", len(train_loader))

Примеров в train: 6951
Батчей за эпоху: 54


In [17]:
# Ячейка 13
# Cмотрим на батч и строим padding-маску
inp, tgt = next(iter(train_loader))
print("inp:", inp.shape, inp.dtype)        # ожидаем [128, 200], int64
print("tgt:", tgt.shape)

pad_mask = (inp != PAD_IDX)                # True = реальный айтем, False = PAD

print("\nОдна строка батча (последние 12 позиций):")
print("inp :", inp[0, -12:].tolist())
print("tgt :", tgt[0, -12:].tolist())
print("mask:", pad_mask[0, -12:].tolist())

print("\nДоля реальных (не-PAD) позиций в батче:",
      f"{pad_mask.float().mean().item():.1%}")

inp: torch.Size([128, 200]) torch.int64
tgt: torch.Size([128, 200])

Одна строка батча (последние 12 позиций):
inp : [11504, 4901, 67478, 86301, 53054, 9996, 73867, 1900, 21546, 8023, 1431, 9722]
tgt : [4901, 67478, 86301, 53054, 9996, 73867, 1900, 21546, 8023, 1431, 9722, 840]
mask: [True, True, True, True, True, True, True, True, True, True, True, True]

Доля реальных (не-PAD) позиций в батче: 41.7%


## 4. Модель: causal self-attention с нуля

In [18]:
# Ячейка 14
import torch, torch.nn as nn, torch.nn.functional as F, math

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, max_len, dropout):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads, self.d_head = n_heads, d_model // n_heads
        self.qkv  = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)
        self.attn_drop, self.resid_drop = nn.Dropout(dropout), nn.Dropout(dropout)
        causal = torch.tril(torch.ones(max_len, max_len)).bool()
        self.register_buffer('causal', causal)

    def forward(self, x, pad_mask):
        B, L, _ = x.shape
        q, k, v = self.qkv(x).split(x.size(-1), dim=-1)
        q = q.view(B, L, self.n_heads, self.d_head).transpose(1, 2)
        k = k.view(B, L, self.n_heads, self.d_head).transpose(1, 2)
        v = v.view(B, L, self.n_heads, self.d_head).transpose(1, 2)
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)
        scores = scores.masked_fill(~self.causal[:L, :L], float('-inf'))
        scores = scores.masked_fill(~pad_mask.view(B, 1, 1, L), float('-inf'))
        attn = F.softmax(scores, dim=-1)
        attn = torch.nan_to_num(attn, nan=0.0)
        attn = self.attn_drop(attn)
        out = (attn @ v).transpose(1, 2).contiguous().view(B, L, -1)
        return self.resid_drop(self.proj(out))

class Block(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, max_len, dropout):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, max_len, dropout)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(),
                                nn.Linear(d_ff, d_model), nn.Dropout(dropout))
    def forward(self, x, pad_mask):
        x = x + self.attn(self.ln1(x), pad_mask)
        x = x + self.ff(self.ln2(x))
        return x

class SASRec(nn.Module):
    def __init__(self, vocab_size, max_len, d_model=64, n_heads=2,
                 n_blocks=2, d_ff=256, dropout=0.2, pad_idx=0):
        super().__init__()
        self.pad_idx, self.max_len = pad_idx, max_len
        self.item_emb = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        self.pos_emb  = nn.Embedding(max_len, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([Block(d_model, n_heads, d_ff, max_len, dropout)
                                     for _ in range(n_blocks)])
        self.ln_f = nn.LayerNorm(d_model)
        self.apply(self._init_weights)
        with torch.no_grad():
            self.item_emb.weight[self.pad_idx].fill_(0)

    def _init_weights(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.zeros_(m.bias)

    def forward(self, seq):
        B, L = seq.shape
        pad_mask = (seq != self.pad_idx)
        pos = torch.arange(L, device=seq.device).unsqueeze(0)
        x = self.drop(self.item_emb(seq) + self.pos_emb(pos))
        for blk in self.blocks:
            x = blk(x, pad_mask)
        return self.ln_f(x)

In [19]:
# Ячейка 15
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = SASRec(vocab_size=vocab_size, max_len=MAX_LEN,
               d_model=64, n_heads=2, n_blocks=2, d_ff=256, dropout=0.2).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Всего параметров: {n_params:,}")
print(f"  в т.ч. item-эмбеддинги: {model.item_emb.weight.numel():,}")

inp, tgt = next(iter(train_loader))
with torch.no_grad():
    h = model(inp.to(device))

print("\nвход :", tuple(inp.shape))
print("выход:", tuple(h.shape), "- должно быть (128, 200, 64)")
print("NaN в выходе:", torch.isnan(h).any().item(), "- должно быть False")

Всего параметров: 11,693,248
  в т.ч. item-эмбеддинги: 11,580,352

вход : (128, 200)
выход: (128, 200, 64) - должно быть (128, 200, 64)
NaN в выходе: False - должно быть False


## 5. Loss с negative sampling

In [20]:
# Ячейка 16 loss с negative sampling
def sasrec_loss(model, seq, tgt, pad_idx=0):
    h = model(seq)                                   # [B, L, D]
    vocab = model.item_emb.num_embeddings

    pos_emb = model.item_emb(tgt)                    # [B,L,D] эмбеддинги настоящих следующих треков
    neg = torch.randint(1, vocab, tgt.shape, device=seq.device)
    neg_emb = model.item_emb(neg)                    # [B,L,D]

    pos_logits = (h * pos_emb).sum(-1)               # [B,L] score настоящего трека
    neg_logits = (h * neg_emb).sum(-1)               # [B,L] score случайной подделки

    mask = (tgt != pad_idx).float()                  # loss только на реальных позициях, PAD игнорируем
    pos_l = F.binary_cross_entropy_with_logits(pos_logits, torch.ones_like(pos_logits),  reduction='none')
    neg_l = F.binary_cross_entropy_with_logits(neg_logits, torch.zeros_like(neg_logits), reduction='none')
    return ((pos_l + neg_l) * mask).sum() / mask.sum()

In [21]:
# Ячейка 17 проверка начального loss
torch.manual_seed(0)
model = SASRec(vocab_size=vocab_size, max_len=MAX_LEN).to(device)
model.train()

inp, tgt = next(iter(train_loader))
loss = sasrec_loss(model, inp.to(device), tgt.to(device), PAD_IDX)
print(f"Начальный loss: {loss.item():.4f}")
print("Ожидаем около 1.386 (= 2 · ln 2)")

Начальный loss: 1.3917
Ожидаем около 1.386 (= 2 · ln 2)


## 6. Обучение с чекпойнтами на Drive

In [22]:
# Ячейка 18 чекпоинты
import os

CKPT_PATH = f"{PROJECT_DIR}/sasrec_likes.pt"

def save_ckpt(model, optimizer, epoch, path=CKPT_PATH):
    torch.save({'model': model.state_dict(),
                'optim': optimizer.state_dict(),
                'epoch': epoch}, path)

def load_ckpt(model, optimizer, path=CKPT_PATH):
    if not os.path.exists(path):
        print("Чекпойнта нет - начинаем с нуля")
        return 0        # стартовая эпоха
    ck = torch.load(path, map_location=device)
    model.load_state_dict(ck['model'])
    optimizer.load_state_dict(ck['optim'])
    print(f"Загружен чекпойнт после эпохи {ck['epoch']}")
    return ck['epoch']  # продолжим со следующей

In [23]:
# Ячейка 19
torch.manual_seed(0)
model = SASRec(vocab_size=vocab_size, max_len=MAX_LEN).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)

start_epoch = load_ckpt(model, optimizer) # 0, если чекпойнта ещё нет, иначе - продолжаем

Загружен чекпойнт после эпохи 20


In [24]:
# Ячейка 20
N_EPOCHS = 20

for epoch in range(start_epoch, N_EPOCHS):
    model.train()
    running = 0.0
    for inp, tgt in train_loader:
        inp, tgt = inp.to(device), tgt.to(device)
        optimizer.zero_grad()                 # 1. обнулить градиенты
        loss = sasrec_loss(model, inp, tgt, PAD_IDX)
        loss.backward()                       # 2. посчитать градиенты
        optimizer.step()                      # 3. шаг оптимизатора
        running += loss.item()

    avg = running / len(train_loader)
    save_ckpt(model, optimizer, epoch + 1)    # чекпойнт КАЖДУЮ(!!!) эпоху
    print(f"эпоха {epoch+1:2d}/{N_EPOCHS}  |  train loss {avg:.4f}")

## 7. Оценка: метрики и baseline'ы

In [25]:
# Ячейка 21
import numpy as np

N_NEG = 100
K = 10

def fair_rank(scores, target_idx=0):
    # 0-индексный ранг настоящего трека; ничьи делим поровну, а не отдаём ему даром
    greater = int((scores > scores[target_idx]).sum())    # строго выше настоящего
    equal   = int((scores == scores[target_idx]).sum())   # равны ему (вкл. сам трек, >=1)
    return greater + (equal - 1) / 2.0

def evaluate(model, eval_data, vocab_size, max_len, pad_idx=0, seed=0):
    model.eval()
    rng = np.random.default_rng(seed)
    ndcg = hit = n = 0
    with torch.no_grad():
        for uid, (inp_seq, target) in eval_data.items():
            seq = inp_seq[-max_len:]
            seq = [pad_idx] * (max_len - len(seq)) + seq
            h_last = model(torch.tensor([seq], device=device))[0, -1]

            negs = rng.integers(1, vocab_size, size=N_NEG).tolist()
            cand = torch.tensor([target] + negs, device=device)
            scores = model.item_emb(cand) @ h_last

            rank = fair_rank(scores)
            if rank < K:
                hit += 1
                ndcg += 1.0 / np.log2(rank + 2)
            n += 1
    # Recall@10 == HitRate@10 при одном релевантном айтеме
    return {'NDCG@10': ndcg/n, 'HitRate@10': hit/n, 'Recall@10': hit/n, 'users': n}

sasrec_test = evaluate(model, test_data, vocab_size, MAX_LEN, PAD_IDX)
sasrec_val  = evaluate(model, val_data,  vocab_size, MAX_LEN, PAD_IDX)
print("SASRec test:", {k: round(v, 4) for k, v in sasrec_test.items()})
print("SASRec val :", {k: round(v, 4) for k, v in sasrec_val.items()})

SASRec test: {'NDCG@10': np.float64(0.4237), 'HitRate@10': 0.6349, 'Recall@10': 0.6349, 'users': 6951}
SASRec val : {'NDCG@10': np.float64(0.4358), 'HitRate@10': 0.6464, 'Recall@10': 0.6464, 'users': 6951}


In [26]:
# Ячейка 22
# baseline'ы на том же протоколе + итоговая таблица
from collections import Counter

pop = Counter()
for seq in train_seqs.values():           # частоты ТОЛЬКО по train, без утечки из val/test
    pop.update(seq)
pop_rank = {it: r for r, (it, _) in enumerate(pop.most_common())}   # 0 = самый популярный
MAX_R = len(pop_rank)

def eval_baselines(eval_data, vocab_size, seed=0):
    rng = np.random.default_rng(seed)
    out = {'popularity': [0.0, 0], 'last_item': [0.0, 0]}   # [сумма ndcg, число hits]
    n = 0
    for uid, (inp_seq, target) in eval_data.items():
        negs = rng.integers(1, vocab_size, size=N_NEG).tolist()
        cand = [target] + negs

        pop_scores = np.array([-pop_rank.get(c, MAX_R) for c in cand])   # редкий трек -> в конец
        rank = fair_rank(pop_scores)
        if rank < K:
            out['popularity'][0] += 1.0/np.log2(rank+2); out['popularity'][1] += 1

        last = inp_seq[-1]
        li_scores = np.array([1 if c == last else 0 for c in cand])      # балл только последнему треку
        rank = fair_rank(li_scores)
        if rank < K:
            out['last_item'][0] += 1.0/np.log2(rank+2); out['last_item'][1] += 1
        n += 1
    return {name: {'NDCG@10': nd/n, 'HitRate@10': ht/n} for name, (nd, ht) in out.items()}

base = eval_baselines(test_data, vocab_size)

print(f"{'модель':<13}{'NDCG@10':>10}{'HitRate@10':>13}")
print("-" * 36)
for name, m in [('last_item',  base['last_item']),
                ('popularity', base['popularity']),
                ('SASRec',      sasrec_test)]:
    print(f"{name:<13}{m['NDCG@10']:>10.4f}{m['HitRate@10']:>13.4f}")

модель          NDCG@10   HitRate@10
------------------------------------
last_item        0.0145       0.0145
popularity       0.3975       0.6119
SASRec           0.4237       0.6349


## Демка: рекомендации для случайного юзера

In [28]:
# Ячейка 23
# Cлучайный человекочитаемый пример (даёт разный результат каждый запуск)
import random
model.eval()

PIN_UID = None     # uid (например 100), чтобы зафиксировать; None = случайный
TOPK = 5

# случайный юзер с историей хотя бы 5 треков
eligible = [u for u, (inp, _) in test_data.items() if len(inp) >= 5]
example_uid = PIN_UID if PIN_UID is not None else random.choice(eligible)

inp_idx, true_next_idx = test_data[example_uid]
seed_idx = inp_idx[-5:]

seq = inp_idx[-MAX_LEN:]
seq = [PAD_IDX] * (MAX_LEN - len(seq)) + seq
seq_t = torch.tensor([seq], device=device)

with torch.no_grad():
    h_last = model(seq_t)[0, -1]
    raw = model.item_emb.weight @ h_last
    raw[PAD_IDX] = -float('inf')

true_rank = int((raw > raw[true_next_idx]).sum())

rec = raw.clone()
for s in set(inp_idx):
    rec[s] = -float('inf')
top = torch.topk(rec, TOPK)

print(f"Пользователь uid = {example_uid}  (история: {len(inp_idx)} треков)")
print("=" * 52)
print("Затравка - последние 5 прослушанных (item_id):")
for i, ix in enumerate(seed_idx, 1):
    print(f"   {i}. трек {idx2item[ix]}")

print(f"\nМодель рекомендует следующими (топ-{TOPK} из всего каталога):")
for r, (sc, ix) in enumerate(zip(top.values.tolist(), top.indices.tolist()), 1):
    print(f"   {r}. трек {idx2item[ix]:<10}  (score {sc:+.2f})")

print("-" * 52)
if true_rank < 10:
    verdict = "- попал в топ-10 каталога!"
else:
    pct = (true_rank + 1) / n_items * 100
    verdict = f"- это топ-{pct:.1f}% каталога"
print(f"А реально следующим был трек {idx2item[true_next_idx]}")
print(f"Модель поставила его на #{true_rank + 1:,} из {n_items:,} треков {verdict}")

Пользователь uid = 240800  (история: 87 треков)
Затравка - последние 5 прослушанных (item_id):
   1. трек 630668
   2. трек 827063
   3. трек 4848997
   4. трек 5910937
   5. трек 3446194

Модель рекомендует следующими (топ-5 из всего каталога):
   1. трек 3542184     (score +7.04)
   2. трек 159001      (score +6.90)
   3. трек 5635052     (score +6.53)
   4. трек 4046840     (score +6.44)
   5. трек 4242661     (score +6.44)
----------------------------------------------------
А реально следующим был трек 293448
Модель поставила его на #2,798 из 180,942 треков - это топ-1.5% каталога


In [31]:
# Утилита: вычищает виджет-метаданные из ipynb перед публикацией на GitHub
import json

NB_PATH = "/content/drive/MyDrive/Colab Notebooks/SASRec.ipynb"  # путь к самому ноутбуку
# (если не уверен, где он лежит — выполни сначала !ls "/content/drive/MyDrive/Colab Notebooks/")

with open(NB_PATH) as f:
    nb = json.load(f)

# 1) убрать widget-state на уровне всего ноутбука
removed = nb.get('metadata', {}).pop('widgets', None)

# 2) убрать widget-view из outputs каждой ячейки
cleaned = 0
for c in nb.get('cells', []):
    for out in c.get('outputs', []):
        if isinstance(out.get('data'), dict):
            if out['data'].pop('application/vnd.jupyter.widget-view+json', None) is not None:
                cleaned += 1

with open(NB_PATH, 'w') as f:
    json.dump(nb, f, ensure_ascii=False, indent=1)

print(f"Удалён top-level widgets-блок: {bool(removed)}")
print(f"Прочищено виджет-выводов в ячейках: {cleaned}")

Удалён top-level widgets-блок: False
Прочищено виджет-выводов в ячейках: 0
